In [ ]:

import torch
from torch.autograd.forward_ad import dual_level, make_dual, unpack_dual

class MixedModeSegment(torch.autograd.Function):
    """
    A custom autograd function where:
    1. Part A uses Forward-Mode AD to get a JVP.
    2. Part B uses standard Backward-Mode AD.
    """
    @staticmethod
    def forward(ctx, x, tangent, model_part_a, model_part_b):
        # Save models/state for backward
        ctx.model_part_b = model_part_b
        
        # --- PART A: Forward-Mode AD ---
        with dual_level():
            dual_input = make_dual(x, tangent)
            dual_output_a = model_part_a(dual_input)
            primal_a, jvp_a = unpack_dual(dual_output_a)
        
        # Save primal_a because we need it for Part B's backward pass
        ctx.save_for_backward(primal_a)
        
        # --- PART B: Standard Forward Pass ---
        # Note: We are still in the 'forward' of the autograd.Function
        out = model_part_b(primal_a)
        
        # We return the final output and the JVP from Part A
        return out, jvp_a

    @staticmethod
    def backward(ctx, grad_output, grad_jvp):
        # grad_output is dLoss/dOut
        # grad_jvp is dLoss/dJVP (usually 0 if JVP isn't in the loss)
        
        primal_a, = ctx.saved_tensors
        
        # Re-run Part B with gradient tracking to get dOut/dPrimal_A
        with torch.enable_grad():
            tmp_a = primal_a.detach().requires_grad_(True)
            tmp_out = ctx.model_part_b(tmp_a)
            
            # Backprop from the final output to the boundary between A and B
            grad_at_boundary = torch.autograd.grad(tmp_out, tmp_a, grad_output)[0]
            
        # Return grads for (x, tangent, model_a, model_b)
        # We only care about x and tangent here
        return grad_at_boundary, None, None, None

# Wrapper for ease of use
class MixedADModel(torch.nn.Module):
    def __init__(self, part_a, part_b):
        super().__init__()
        self.part_a = part_a
        self.part_b = part_b

    def forward(self, x, tangent):
        return MixedModeSegment.apply(x, tangent, self.part_a, self.part_b)